# What is wrong with going long the top 1% and short the bottom 1%?

<!-- Too much uncertainty -->


---
🌟🌟🌟 Welcome to Harold's Quant Channel. 🌟🌟🌟


## Factor in Practice Part 4
# Build your own hedge fund: implementing a systematic advanced hedging strategy in Python (2)

##### "I do not know what the future holds, but I know where I will be waiting for it." - Warren Buffett

### 🎬 Host: Director Harold
- 🏛 CUHK-Shenzhen Financial Engineering Undergraduate Program
- 📈 On the path to a U.S. Master's in Financial Engineering (already admitted)
- 🌐 [Follow my Bilibili channel for quant learning content that is easy to follow](https://space.bilibili.com/629573485)
- 📚 [Follow my YouTube channel for more quant investing content](https://www.youtube.com/@BD_Harold)
- 📱 WeChat Official Account: Director Harold. All of the data can be obtained there.

🌟🌟🌟 Quant is not some mystical cure-all. It is a tool that helps retail investors understand markets. #HaroldQuantChannel 🌟🌟🌟

---


Load data


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

# Use a representative subset of S&P 500 stocks
sp500_tickers = [
    'AAPL', 'MSFT', 'AMZN', 'GOOGL', 'META', 'TSLA', 'BRK-B', 'JNJ', 'V', 'WMT',
    'JPM', 'PG', 'MA', 'UNH', 'HD', 'DIS', 'BAC', 'NVDA', 'ADBE', 'CRM',
    'NFLX', 'PYPL', 'INTC', 'VZ', 'T', 'MRK', 'PFE', 'ABT', 'TMO', 'NKE',
    'XOM', 'CVX', 'LLY', 'MCD', 'KO', 'PEP', 'ABBV', 'AVGO', 'COST', 'ACN',
    'TXN', 'DHR', 'NEE', 'UPS', 'PM', 'LIN', 'LOW', 'QCOM', 'SPGI', 'IBM',
    'BMY', 'AMGN', 'GS', 'BLK', 'CAT', 'GE', 'MMM', 'AXP', 'SCHW', 'BKNG'
]

start = '2005-01-01'
end = '2022-12-31'

monthly_data = []
for ticker in sp500_tickers:
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(start=start, end=end, interval='1mo')
        hist = hist.reset_index()
        hist['stock'] = ticker
        hist['date'] = hist['Date'].dt.strftime('%Y-%m')
        hist['monthly_return'] = hist['Close'].pct_change()
        info = stock.info
        hist['market_cap'] = hist['Close'] * info.get('sharesOutstanding', 1e9)
        hist['value_in_thousand'] = hist['market_cap'] / 1000
        hist['next_month_return'] = hist['monthly_return'].shift(-1)
        monthly_data.append(hist[['stock', 'date', 'value_in_thousand', 'monthly_return', 'next_month_return']])
    except:
        pass

df_mkt = pd.concat(monthly_data, axis=0).dropna(subset=['monthly_return']).reset_index(drop=True)
unique_dates = df_mkt['date'].unique()
unique_dates.sort()
unique_dates_nochange = unique_dates.copy()
unique_dates = [datetime.strptime(date, "%Y-%m") for date in unique_dates]
df_mkt

---
# What else do we still need to think about?
## What problems might show up? Is this stock-selection rule too aggressive? Can we derive a conclusion that works across the full market instead of only at the extremes?

---


In [ ]:
# Initialize return lists for each portfolio
df = df_mkt
portfolio_returns = {f'portfolio_{i}': [] for i in range(1, 6)}

for date in unique_dates_nochange:

    df_date = df[df['date'] == date]
    
    # Check if df_date is empty
    if df_date.empty:
        # If empty, skip this date
        continue
    
    # Get quintile thresholds
    thresholds = np.percentile(df_date['value_in_thousand'], [20, 40, 60, 80, 100])
    
    # Build five portfolios
    for i in range(5):
        if i == 0:
            # First portfolio starts from minimum
            portfolio = df_date[df_date['value_in_thousand'] <= thresholds[i]]
        else:
            # Subsequent portfolios between two thresholds
            portfolio = df_date[(df_date['value_in_thousand'] > thresholds[i-1]) & (df_date['value_in_thousand'] <= thresholds[i])]
        
        # If portfolio is empty, continue to next
        if portfolio.empty:
            portfolio_return = 0  # 或者你可以选择跳过这个组合
        else:
            # Here we still look at next month's return
            if not np.isnan(portfolio['next_month_return'].mean()):
                portfolio_return = portfolio['next_month_return'].mean()

        portfolio_returns[f'portfolio_{i+1}'].append(portfolio_return)


In [ ]:
# Hedge portfolio return is long portfolio 1 minus short portfolio 5
hedge_returns = []
for i in range(len(portfolio_returns['portfolio_1'])):
    hedge_return = portfolio_returns['portfolio_1'][i] - portfolio_returns['portfolio_5'][i]
    hedge_returns.append(hedge_return)

# Output average return for each portfolio
average_returns = {portfolio: np.mean(returns) for portfolio, returns in portfolio_returns.items()}
average_hedge_return = np.mean(hedge_returns)

# Print results
print("Average Returns by Portfolio:")
for portfolio, average_return in average_returns.items():
    print(f"{portfolio}: {average_return}")

print(f"Average Hedge Return: {average_hedge_return}")

In [ ]:
# Calculate the cumulative returns for each portfolio
cumulative_portfolio_returns = {portfolio: np.cumsum(returns) for portfolio, returns in portfolio_returns.items()}
# Calculate the cumulative hedge returns
cumulative_hedge_returns = np.cumsum(hedge_returns)
# Get the final cumulative return for each portfolio and the hedge strategy
final_cumulative_returns = {portfolio: returns[-1] if len(returns) > 0 else 0 for portfolio, returns in cumulative_portfolio_returns.items()}
final_cumulative_hedge_return = cumulative_hedge_returns[-1] if len(cumulative_hedge_returns) > 0 else 0
# Print the final cumulative returns by portfolio
print("Final Cumulative Returns by Portfolio:")
for portfolio, final_return in final_cumulative_returns.items():
    print(f"{portfolio}: {final_return}")
# Print the final cumulative hedge return
print(f"Final Cumulative Hedge Return: {final_cumulative_hedge_return}")


In [ ]:
# Calculate the cumulative returns for each portfolio
cumulative_portfolio_returns = {portfolio: np.cumsum(returns) for portfolio, returns in portfolio_returns.items()}

# Plotting the cumulative returns for each portfolio
plt.figure(figsize=(14, 7))  # Set the figure size

for portfolio, returns in cumulative_portfolio_returns.items():
    plt.plot(unique_dates, returns, label=portfolio)

plt.title('Cumulative Portfolio Returns Over Time')
plt.xlabel('Time Periods')
plt.ylabel('Cumulative Returns')
plt.legend()
plt.show()

---

# Factors...?

Harold, didn't you say that a factor can help determine whether a stock goes up or down?

---
